## Modularize file format converter

In [44]:
import os
import pandas as pd 
import json 
import re
import glob
from pathlib import Path

In [45]:
def get_column_names(schemas, ds_name, sorting_key='column_position'):
    columns_details = schemas[ds_name]
    columns = sorted(columns_details, key=lambda col: col[sorting_key])
    return [col['column_name'] for col in columns]

In [46]:
def read_csv(file, schemas):
    file_path_list = re.split('[/\\\]', file)

    ds_name = file_path_list[-2]
    # file_name = re.split('r[/\\]', file)[-1]

    columns = get_column_names(schemas, ds_name)

    df = pd.read_csv(file, names=columns)

    return df 

<>:2: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<>:2: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
C:\Users\Asus\AppData\Local\Temp\ipykernel_26188\1974826611.py:2: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
  file_path_list = re.split('[/\\\]', file)


In [47]:
def to_json(df, tgt_base_dir, ds_name, file_name):
    json_file_path = f'{tgt_base_dir}/{ds_name}/{file_name}'
    os.makedirs(f'{tgt_base_dir}/{ds_name}' , exist_ok=True)
    df.to_json(
        json_file_path,
        orient='records',
        lines=True
    )

In [48]:
def file_converter(ds_name):
    src_base_dir = '../../../resources/data/retail_db'
    tgt_base_dir = '../../../resources/data/retail_db_json'

    schemas = json.load(open(f'{src_base_dir}/schemas.json'))
    files = glob.glob(f'{src_base_dir}/{ds_name}/part-*')

    for file in files:
        print(f'Processing {file}')
        
        df = read_csv(file, schemas)
        # file_name = re.split(r'[/\\]', file)[-1]
        file_name = os.path.basename(file)
        to_json(df, tgt_base_dir, ds_name, file_name)

### Try to use function

In [49]:
ds_name = 'orders'

file_converter(ds_name)

Processing ../../../resources/data/retail_db/orders\part-00000


## Wraper to process all data set

In [54]:
def file_converter(src_base_dir, tgt_base_dir, ds_name):
    schemas = json.load(open(f'{src_base_dir}/schemas.json'))
    files = glob.glob(f'{src_base_dir}/{ds_name}/part-*')

    for file in files:
        df = read_csv(file, schemas)
        file_name = os.path.basename(file)
        to_json(df, tgt_base_dir, ds_name, file_name)

In [55]:
def process_files(ds_names=None):
    src_base_dir = '../../../resources/data/retail_db'
    tgt_base_dir = '../../../resources/data/retail_db_json'

    schemas = json.load(open(f'{src_base_dir}/schemas.json'))
    
    if not ds_names:
        ds_name = schemas.keys()

    for ds_name in ds_names:
        print(f'Processing {ds_name}')
        file_converter(src_base_dir, tgt_base_dir, ds_name)

In [56]:
process_files(['orders', 'order_items'])

Processing orders
Processing order_items


In [57]:
schemas = json.load(open('../../../resources/data/retail_db/schemas.json'))

schemas.keys()

dict_keys(['departments', 'categories', 'orders', 'products', 'customers', 'order_items'])